# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [17]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este o masina."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este o masina."
)

print(raspuns)

O mașină este un vehicul autopropulsat, de obicei cu patru roți, folosit pentru transport personal sau de marfă. Aceasta funcționează pe baza unui motor care transformă energia, de obicei din combustibili fosili sau electricitate, în mișcare.
O mașină este un vehicul motorizat cu roți, conceput pentru a transporta pasageri sau mărfuri pe drumuri. Aceasta funcționează prin intermediul unui motor care transformă energia (de obicei chimică din combustibil) în mișcare.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.6, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [7]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, principalele consecinte din politica românească dupa anularea turului 1 al ultimelor alegeri.
Maximum 80 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
Anularea turului 1 al alegerilor a condus la o criză politică și la o reconfigurare a alianțelor, determinând noi negocieri pentru formarea guvernului. Această situație a generat incertitudine și a influențat agenda legislativă, punând accent pe stabilizarea politică și pe organizarea rapidă a unui nou scrutin.

--- Gemini 2.5 Flash ---
Anularea turului 1 al alegerilor locale a transformat scrutinul într-unul cu un singur tur, favorizând semnificativ candidații în funcție și partidele mari. Această modificare a impus partidelor o regândire a strategiilor electorale și a alianțelor pre-electorale pentru a obține majoritatea simplă necesară.

--- OpenRouter Free ---
<｜begin▁of▁sentence｜><|endoftext|>


## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [8]:
SYSTEM = """
Esti un cercetator politic care analizeaza discursul anti-suveranist din media. 
Raspunzi scurt, clar si strict pe baza informatiilor pe care le ai, fara sa inventezi.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii fură, iar oamenii simpli plătesc din buzunarul propriu. Doar nevoiile celor de la putere conteaza."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.2
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Acuzator, resentimentar.
Emoție dominantă: Furie, frustrare.
Țintă principală: Clasa politică, elita.
Populism: da

--- Gemini 2.5 Flash ---
Ton: Critic, acuzator, resentimentar.
Emoție dominantă: Frustrare, furie, sentiment de nedreptate.
Țintă principală: Clasa politică / Cei de la putere.
Populism: da

--- OpenRouter Free ---
Emoție dominantă este rauza.  
Principă este critica a elitism.  
Populism este negativ.  
Concluzie: răspuns este negativ.


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [9]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "fericire", "speranta", "dezamagire", "ironie", "frustrare", "nedreptate"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [10]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc din buzunarul propriu. Doar nevoiile celor de la putere conteaza"

SYSTEM = "Esti un cercetator politic care analizeaza discursul anti-suveranist din media. "

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'negativ', 'emotie_dominanta': 'nedreptate', 'tinta_principala': 'politicieni', 'populism': True, 'explicatie_scurta': 'Comentariul exprimă o nemulțumire generală față de clasa politică, sugerând corupție și nepăsare față de cetățeanul de rând, un discurs tipic populist.'}

--- Gemini 2.5 Flash ---
{'ton': 'negativ', 'emotie_dominanta': 'nedreptate', 'tinta_principala': 'Clasa politică și de la putere', 'populism': True, 'explicatie_scurta': "Comentariul exprimă o critică vehementă la adresa clasei politice, acuzând-o de corupție și indiferență față de cetățenii obișnuiți, un discurs tipic populist care opune 'oamenii simpli' elitei corupte."}

--- OpenRouter Free ---
[Eroare API: Error code: 404 - {'error': {'message': 'Provider returned error', 'code': 404, 'metadata': {'raw': '{"status":404,"title":"Not Found","detail":"Function id \'74819e7c-e3e6-4497-8fdb-5f5fdc17dc85\' version \'null\': Specified function in account \'sxpm0VR-knE2_K2__i14Zbz

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [18]:
PROMPT_STAB = """
CCR a anulat alegerile, posibil din cauza unor interferente ruse.
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.2, 0.8, 1.4]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.2:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

temperature=0.8:
Anularea alegerilor de către CCR, în contextul unor posibile interferențe rusești, poate duce la incertitudine politică și la o perioadă de instabilitate. Această situație ar putea influența procesul democratic și ar putea genera dezbateri cu privire la integritatea proceselor electorale.

temperature=1.4:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

[ Gemini 2.5 Flash ]

temperature=0.2:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

temperature=0.8:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

temperature=1.4:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

[ OpenRouter Free ]

temperature=0.2:
Anularea alegerilor de către CCR, cu posibile suspiciuni de interferențe ruse, poate genera instabilitate politică prin suspendarea procesulu

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da | da | da | partial | Stabil lingvistic, dar a returnat eroari|
| Gemini 2.5 Flash | da | da | da | da | Raspunde corect in romana, stabil la temperaturi mari|
| OpenRouter Free |  nu | partial | nu | da | Produce halucinatii, abereaza, nu raspunde corect in romana, a returnat o eroare|
### Decizie
**Model principal ales: Gemini 2.5 Flash Lite**  
**Model de rezervă: Gemini 2.5 Flash**  
**Temperature recomandată: 0.2**  
**De ce am ales acest model?**  
Am ales acest model deoarece este singurul care la prima executare nu a returnat erori de server(desi dupa o a doua executare a intampinat erori de quota), a respectat cerintele impuse si a formulat corect in limba romana. Raspunsurile primite aveau o logica, comparativ  cu cele ale OpenRouter care erau alcatuite din aberatii scrise gresit. Raspunsurile generate de Gemini au fost stabile, ideea principala regasindu-se corect in cazul oricarei temperaturi, modelul fiind potrivit pentru adnotarea comentariilor.


## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [17]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales